# GTSAM PPP-RTK (QZSS CLAS) with integer ambiguity resolution

Gtsam-first **PPP-RTK** step by step. cssrlib decodes QZSS CLAS (L6) and prepares
undifferenced, SSR-corrected residuals; GTSAM's `Undifferenced{Pseudorange,
CarrierPhase}Factor` (inuex35/gtsam fork) estimate a static position together with
per-epoch per-system **clock**, per-epoch tropospheric **ZTD** (random walk),
per-satellite slant **ionosphere** (tight CLAS prior) and float **ambiguities**,
updated with **incremental ISAM2**; integers are resolved with cssrlib LAMBDA via
the `gnss_ar` bridge.

Needs the companion `cssrlib-data` repo (CLAS data + ANTEX) and the custom gtsam
build with the Undifferenced* factors.

In [1]:
import os
from copy import deepcopy
from binascii import unhexlify
import numpy as np

from cssrlib.cssrlib import cssr
from cssrlib.gnss import (ecef2pos, ecef2enu, Nav, time2gpst, time2doy,
                          rSigRnx, epoch2time, sat2prn, uGNSS)
from cssrlib.gnss import geodist as cssr_geodist
from cssrlib.peph import atxdec, searchpcv
from cssrlib.ppprtk import ppprtkpos
from cssrlib.rinex import rnxdec
import gtsam
from gtsam import symbol

from gnss_ar import resolve_ar

SYSS = (uGNSS.GPS, uGNSS.GAL, uGNSS.QZS)
X = symbol('x', 0)
def CK(ei, si): return symbol('c', ei * 4 + si)   # per-epoch, per-system clock
def ZT(ei): return symbol('z', ei)                # per-epoch ZTD (random walk)
def IO(s): return symbol('i', int(s))             # per-sat slant iono
def AM(s, f): return symbol('n', int(s) * 4 + f)  # per-sat/freq ambiguity

## 1. Load RINEX, ANTEX and the CLAS grid

`ppprtkpos` is the cssrlib PPP-RTK front-end. `xyz_ref` is the surveyed marker
(accuracy scoring only).

In [2]:
DATADIR = os.environ.get('CSSRLIB_DATA',
    os.path.join(os.path.dirname(os.getcwd()), 'cssrlib-data', 'data'))
ep = [2025, 8, 21, 7, 0, 0]
xyz_ref = np.array([-3962108.7007, 3381309.5532, 3668678.6648])
pos_ref = ecef2pos(xyz_ref)
NEP = int(os.environ.get('NEP', '120'))
time = epoch2time(ep); doy = int(time2doy(time)); let = chr(ord('a') + ep[3])
bdir = f'{DATADIR}/doy{ep[0]:04d}-{doy:03d}/'

sigs = [rSigRnx("GC1C"), rSigRnx("GC2W"), rSigRnx("EC1C"), rSigRnx("EC5Q"),
        rSigRnx("JC1C"), rSigRnx("JC2L"),
        rSigRnx("GL1C"), rSigRnx("GL2W"), rSigRnx("EL1C"), rSigRnx("EL5Q"),
        rSigRnx("JL1C"), rSigRnx("JL2L"),
        rSigRnx("GS1C"), rSigRnx("GS2W"), rSigRnx("ES1C"), rSigRnx("ES5Q"),
        rSigRnx("JS1C"), rSigRnx("JS2L")]
nav = Nav(); nav = rnxdec().decode_nav(bdir + f'{doy:03d}{let}_rnx.nav', nav)
atx = atxdec(); atx.readpcv(f'{DATADIR}/antex/igs20.atx')
rnx = rnxdec(); rnx.setSignals(sigs)
cs = cssr(); cs.monlevel = 0; cs.week = time2gpst(time)[0]
cs.read_griddef(f'{DATADIR}/clas_grid.def')
assert rnx.decode_obsh(bdir + f'{doy:03d}{let}_rnx.obs') >= 0
rnx.autoSubstituteSignals()
ppp = ppprtkpos(nav, rnx.pos)
nav.rcv_ant = searchpcv(atx.pcvr, rnx.ant, rnx.ts); nav.sat_ant = atx.pcvs
cs.find_grid_index(ecef2pos(rnx.pos)); nf = nav.nf
print('frequencies:', nf, ' constellations:', [int(s) for s in SYSS])

frequencies: 2  constellations: [0, 1, 2]


## 2. Decode CLAS (L6) and build the front-end measurements

For each epoch the L6 stream is decoded; once a full CLAS set is available,
`prepare_ppp_measurements` returns SSR-corrected undifferenced residuals (`y`),
satellite states (`rs`), tropo mapping (`mapfw`), iono coefficients (`mu`),
wavelengths (`lam`) and the CLAS atmosphere a-priori sigmas (`iono_sig`/`ztd_sig`).

In [3]:
v = np.genfromtxt(bdir + f'{doy:03d}{let}_qzsl6.txt',
                  dtype=[('wn', 'int'), ('tow', 'int'), ('prn', 'int'),
                         ('type', 'int'), ('len', 'int'), ('nav', 'S500')])
frames = []
obs = rnx.decode_obs()
while time > obs.t and obs.t.time != 0:
    obs = rnx.decode_obs()
for k in range(NEP):
    week, tow = time2gpst(obs.t)
    vi = v[(v['tow'] == tow) & (v['type'] == 0) & (v['prn'] == 199)]
    if len(vi) > 0:
        cs.decode_l6msg(unhexlify(vi['nav'][0]), 0)
        if cs.fcnt == 5:
            cs.decode_cssr(bytes(cs.buff), 0)
    if k == 0:
        nav.t = deepcopy(obs.t); t0 = deepcopy(obs.t)
        t0.time = t0.time // 30 * 30; cs.time = obs.t; nav.time_p = t0
    if cs.chk_stat():
        ppm = ppp.prepare_ppp_measurements(obs, cs=cs, pos_pred=rnx.pos)
        if ppm is not None and k >= 20:
            frames.append(ppm)
    obs = rnx.decode_obs()
    if obs.t.time == 0:
        break
print(f'{len(frames)} epochs collected via prepare_ppp_measurements')

100 epochs collected via prepare_ppp_measurements


## 3. State model and the random-walk tropo factor

PPP cannot difference errors away, so they become states: a per-epoch per-system
**clock** `CK(ei, sys)`, a per-epoch **ZTD** `ZT(ei)` linked across epochs by a
random-walk factor, a per-satellite slant **iono** `IO(s)` (tight CLAS STEC prior),
and per-satellite/frequency **ambiguities** `AM(s, f)`. ISAM2 uses **QR**.

In [4]:
params = gtsam.ISAM2Params(); params.setFactorization('QR')
isam = gtsam.ISAM2(params)
x0 = xyz_ref + np.array([5.0, -4.0, 3.0])      # deliberately ~7 m off
ztd_sigs = [fr.ztd_sig for fr in frames if np.isfinite(fr.ztd_sig)]
ztd_sig = float(np.median(ztd_sigs)) if ztd_sigs else 0.1


def ztd_rw():
    def err(this, values, jac):
        a = values.atDouble(this.keys()[0]); b = values.atDouble(this.keys()[1])
        if jac is not None:
            jac[0] = np.array([[1.0]]); jac[1] = np.array([[-1.0]])
        return np.array([a - b])
    return err

## 4. Build the graph incrementally and resolve each epoch

Per epoch: add the ZTD prior (epoch 0) or random-walk link; then for each
satellite/frequency add `UndifferencedPseudorangeFactor` and
`UndifferencedCarrierPhaseFactor` wired to position, the system clock, ZTD, the
slant iono and the ambiguity. We require both frequencies so the per-satellite
iono is observable. After each update we attempt AR with a **convergence gate**
(PPP needs a few epochs before the position is observable).

In [5]:
seen_io, seen_am, seen_ck = set(), set(), set()
first_fix, n_fix, nfac = None, 0, 0
for ei, fr in enumerate(frames):
    graph = gtsam.NonlinearFactorGraph(); val = gtsam.Values()
    if ei == 0:
        val.insert(X, gtsam.Point3(*x0))
        graph.add(gtsam.PriorFactorPoint3(X, gtsam.Point3(*x0),
                  gtsam.noiseModel.Isotropic.Sigma(3, 30.0)))
    rr = fr.pos_pred
    val.insert(ZT(ei), 0.0)
    if ei == 0:
        graph.addPriorDouble(ZT(0), 0.0,
                             gtsam.noiseModel.Isotropic.Sigma(1, ztd_sig))
    else:
        graph.add(gtsam.CustomFactor(
            gtsam.noiseModel.Isotropic.Sigma(1, 0.003),
            gtsam.KeyVector([ZT(ei), ZT(ei - 1)]), ztd_rw()))
    for i, s in enumerate(fr.sat):
        s = int(s); sys = sat2prn(s)[0]
        if sys not in SYSS or fr.el[i] <= 0:
            continue
        if not np.all(np.isfinite(fr.rs[i])) or np.linalg.norm(fr.rs[i]) < 1e6:
            continue
        if not (fr.y[i, 0] != 0 and fr.y[i, 1] != 0
                and fr.y[i, nf] != 0 and fr.y[i, nf + 1] != 0):
            continue
        geom, _ = cssr_geodist(fr.rs[i], rr)
        s_el = 1.0 / max(np.sin(fr.el[i]), 0.1)
        ck = CK(ei, SYSS.index(sys))
        for f in range(nf):
            lam, mu = fr.lam[i, f], fr.mu[i, f]
            if lam <= 0 or mu <= 0 or fr.y[i, f] == 0 or fr.y[i, nf + f] == 0:
                continue
            m_phase = fr.y[i, f] + geom
            m_code = fr.y[i, nf + f] + geom
            if ck not in seen_ck:
                seen_ck.add(ck); val.insert(ck, float(m_code - geom))
                graph.addPriorDouble(ck, 0.0,
                                     gtsam.noiseModel.Isotropic.Sigma(1, 1e5))
            if IO(s) not in seen_io:
                seen_io.add(IO(s)); val.insert(IO(s), 0.0)
                sig_i = min(fr.iono_sig[i] if np.isfinite(fr.iono_sig[i])
                            else 0.05, 0.01)
                graph.addPriorDouble(IO(s), 0.0,
                                     gtsam.noiseModel.Isotropic.Sigma(1, sig_i))
            graph.add(gtsam.UndifferencedPseudorangeFactor(
                X, ck, ZT(ei), IO(s), m_code, gtsam.Point3(*fr.rs[i]),
                fr.mapfw[i], mu, 0.0,
                gtsam.noiseModel.Isotropic.Sigma(1, 0.6 * s_el)))
            ak = AM(s, f)
            if ak not in seen_am:
                seen_am.add(ak)
                val.insert(ak, float((m_phase - geom - val.atDouble(ck)) / lam))
                graph.addPriorDouble(ak, val.atDouble(ak),
                                     gtsam.noiseModel.Isotropic.Sigma(1, 5.0))
            graph.add(gtsam.UndifferencedCarrierPhaseFactor(
                X, ck, ZT(ei), IO(s), ak, m_phase, gtsam.Point3(*fr.rs[i]),
                fr.mapfw[i], mu, lam, 0.0,
                gtsam.noiseModel.Isotropic.Sigma(1, 0.006 * s_el)))
    nfac += graph.size()
    isam.update(graph, val)

    res = isam.calculateEstimate()
    nb, xa = resolve_ar(ppp, isam, res, X, AM, fr.sat, fr.el, seen_am, nf,
                        SYSS, conv_sigma=1.0)
    xh = xa if nb > 0 else np.array(res.atPoint3(X))
    if nb > 0:
        n_fix += 1
        first_fix = ei if first_fix is None else first_fix
    enu = ecef2enu(pos_ref, xh - xyz_ref)
    if ei % 15 == 0 or ei == len(frames) - 1:
        print(f'ep{ei:3d} {"FIX " if nb > 0 else "flt "} nb={nb:2d} '
              f'2D={np.hypot(enu[0], enu[1]):.3f} 3D={np.linalg.norm(xh - xyz_ref):.3f} m')
print(f'\ngraph: {nfac} factors, {len(seen_am)} ambiguities')

ep  0 flt  nb= 0 2D=0.281 3D=0.452 m


ep 15 FIX  nb=18 2D=0.013 3D=0.046 m


ep 30 FIX  nb=18 2D=0.018 3D=0.047 m


ep 45 FIX  nb=18 2D=0.020 3D=0.051 m


ep 60 FIX  nb=22 2D=0.022 3D=0.063 m


ep 75 FIX  nb=22 2D=0.022 3D=0.072 m


ep 90 FIX  nb=22 2D=0.022 3D=0.073 m


ep 99 FIX  nb=22 2D=0.022 3D=0.074 m

graph: 5571 factors, 28 ambiguities


## 5. AR bridge and convergence gate

`resolve_ar` writes the GTSAM float ambiguities + their joint covariance into the
cssrlib `nav` state and runs `resamb_lambda`. As in RTK the full position+ambiguity
joint marginal is ill-conditioned, so the covariance is built from the
ambiguity-only joint plus pairwise (position, ambiguity) cross terms.

Unlike RTK (where pseudorange pins the position directly), in PPP-RTK at the first
epoch the position is co-estimated with clock/ZTD/iono and is not yet observable, so
`conv_sigma=1.0` skips AR until the position 1-sigma drops below ~1 m.

## 6. Results

CLAS PPP-RTK fixes from epoch 1 (after the 1-epoch convergence gate) and converges
to the **cm** level against the surveyed marker -- with a single receiver, no base
station.

In [6]:
print(f'first fix: epoch {first_fix}   fixed {n_fix}/{len(frames)} epochs')

first fix: epoch 1   fixed 99/100 epochs
